In [1]:
import torch
import os
os.chdir('../')

In [2]:
from scipy import linalg
import numpy as np
import os, torch
from tqdm import tqdm
from reports.util import load_config


@torch.no_grad()
def _trace_sqrtm_product(C1: torch.Tensor, C2: torch.Tensor) -> torch.Tensor:
    # Tr sqrtm(C1 @ C2) = Tr sqrt( C1^{1/2} C2 C1^{1/2} )
    s, U = torch.linalg.eigh(C1)                 # C1 = U diag(s) U^T
    s = s.clamp_min(0)
    C1h = (U * s.sqrt()) @ U.t()                 # C1^{1/2}
    M   = C1h @ C2 @ C1h
    w   = torch.linalg.eigvalsh((M + M.t()) * 0.5).clamp_min(0)
    return w.sqrt().sum()

@torch.no_grad()
def calc_fid_stats(mu1, sigma1, mu2, sigma2, eps: float = 1e-6) -> float:
    # 모두 float64 + 동일 device로 정렬
    C1 = torch.as_tensor(sigma1, dtype=torch.float64)
    device = C1.device
    C2 = torch.as_tensor(sigma2, dtype=torch.float64).to(device)
    m1 = torch.as_tensor(mu1,    dtype=torch.float64).to(device).flatten()
    m2 = torch.as_tensor(mu2,    dtype=torch.float64).to(device).flatten()

    D = m1.numel()
    I = torch.eye(D, dtype=torch.float64, device=device)

    # 대칭화 + 정칙화
    C1 = (C1 + C1.t()) * 0.5 + eps * I
    C2 = (C2 + C2.t()) * 0.5 + eps * I

    diff = m1 - m2
    tr_covmean = _trace_sqrtm_product(C1, C2)
    fid = diff.dot(diff) + torch.trace(C1) + torch.trace(C2) - 2.0 * tr_covmean
    return float(fid)

@torch.no_grad()
def calc_fid_pt_dir(pt_dir: str, mu, sigma, eps: float = 1e-6, num=100000, key="inception_feature") -> float:
    # pt_dir에서 'inception_feature'를 모아서 mu1, sigma1 추정 후 FID 계산
    X = []
    for f in tqdm(os.listdir(pt_dir)[:num]):
        if f.endswith(".pt"):
            v = torch.load(os.path.join(pt_dir, f), map_location="cpu").get(key)
            if v is not None:
                X.append(torch.as_tensor(v, dtype=torch.float64).flatten())
    if len(X) < 2:
        raise ValueError("need >=2 features")

    X   = torch.stack(X, 0)                 # [N, D]
    mu1 = X.mean(0)
    Xc  = X - mu1
    sigma1 = (Xc.t() @ Xc) / (X.shape[0] - 1)  # 불편추정

    return calc_fid_stats(mu1, sigma1, mu, sigma, eps=eps)
    #return calculate_frechet_distance(mu1, sigma1, mu, sigma, eps=eps)

def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Numpy implementation of the Frechet Distance.
    The Frechet distance between two multivariate Gaussians X_1 ~ N(mu_1, C_1)
    and X_2 ~ N(mu_2, C_2) is
            d^2 = ||mu_1 - mu_2||^2 + Tr(C_1 + C_2 - 2*sqrt(C_1*C_2)).

    Stable version by Dougal J. Sutherland.

    Params:
    -- mu1   : Numpy array containing the activations of a layer of the
               inception net (like returned by the function 'get_predictions')
               for generated samples.
    -- mu2   : The sample mean over activations, precalculated on an
               representative data set.
    -- sigma1: The covariance matrix over activations for generated samples.
    -- sigma2: The covariance matrix over activations, precalculated on an
               representative data set.

    Returns:
    --   : The Frechet Distance.
    """

    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)

    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert mu1.shape == mu2.shape, \
        'Training and test mean vectors have different lengths'
    assert sigma1.shape == sigma2.shape, \
        'Training and test covariances have different dimensions'

    diff = mu1 - mu2

    # Product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        msg = ('fid calculation produces singular product; '
               'adding %s to diagonal of cov estimates') % eps
        print(msg)
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    # Numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            raise ValueError('Imaginary component {}'.format(m))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)

    return (diff.dot(diff) + np.trace(sigma1)
            + np.trace(sigma2) - 2 * tr_covmean)

In [3]:
pt_dirs = [ #'samplings/GMDiT/1.4/3/DS-Solver_Flow/50000/ds_mobile_0/',
            #'samplings/GMDiT/1.4/5/DS-Solver_Flow/50000/ds_mobile_0/',
            'samplings/GMDiT/1.4/7/DS-Solver_Flow/50000/ds_mobile_0/',
            #'samplings/GMDiT/1.4/9/DS-Solver_Flow/50000/ds_mobile_0/',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(pt_dir, fid)


100%|██████████| 50001/50001 [00:54<00:00, 923.51it/s] 


samplings/GMDiT/1.4/7/DS-Solver_Flow/50000/ds_mobile_0/ 2.8360932705911637


In [18]:
pt_dirs = [ 'samplings/GMDiT/1.4/5/Dual-Solver/50000/mobile_0/',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], num=23481)
    print(pt_dir, fid)


100%|██████████| 23481/23481 [00:15<00:00, 1533.85it/s]


samplings/GMDiT/1.4/5/Dual-Solver/50000/mobile_0/ 3.925477854101757


In [38]:
pt_dirs = [ #'samplings/GMDiT/1.4/3/BNS-Solver/50000/bns_mobile_0/',
            #'samplings/GMDiT/1.4/5/BNS-Solver/50000/bns_mobile_0/'
            #'samplings/GMDiT/1.4/7/BNS-Solver/50000/bns_mobile_0/'
            'samplings/GMDiT/1.4/9/BNS-Solver/50000/bns_mobile_0/'
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(pt_dir, fid)


100%|██████████| 50001/50001 [00:16<00:00, 2955.47it/s]


samplings/GMDiT/1.4/9/BNS-Solver/50000/bns_mobile_0/ 3.0806846971534014


In [31]:
pt_dirs = [ #'samplings/GMDiT/1.4/9/Dual-Solver/50000/mobile_0/',
            #'samplings/GMDiT/1.4/7/Dual-Solver/50000/mobile_0/',
            #'samplings/GMDiT/1.4/5/Dual-Solver/50000/mobile_0/',
            'samplings/GMDiT/1.4/3/Dual-Solver/50000/mobile_0/',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(pt_dir, fid)


 17%|█▋        | 8359/50001 [00:03<00:15, 2620.70it/s]


KeyboardInterrupt: 

In [18]:

pt_dirs = [ #'samplings/GMDiT/1.4/8/DS-Solver_Flow/50000/ds_0',
            #'samplings/GMDiT/1.4/6/DS-Solver_Flow/50000/ds_0',
            'samplings/GMDiT/1.4/4/DS-Solver_Flow/50000/ds_0',
            #'samplings/GMDiT/1.4/8/BNS-Solver/50000/bns_0',            
            #'samplings/GMDiT/1.4/6/BNS-Solver/50000/bns_0',            
            'samplings/GMDiT/1.4/4/BNS-Solver/50000/bns_0',           
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(pt_dir, fid)


100%|██████████| 50001/50001 [00:31<00:00, 1605.99it/s]


samplings/GMDiT/1.4/4/DS-Solver_Flow/50000/ds_0 11.605349891818946


100%|██████████| 50001/50001 [00:31<00:00, 1601.03it/s]


samplings/GMDiT/1.4/4/BNS-Solver/50000/bns_0 26.645228115490795


In [6]:
pt_dirs = [ 
            #'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_0/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_1/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_2/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_3/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_4/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_5/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_6/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_7/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_8/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_9/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_10/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_11/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_12/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_13/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_14/which_0',
             #'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_15/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_16/which_0',
            # 'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_17/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_18/which_0',
            'samplings/GMDiT/1.4/3/Dual-Solver/50000/which_19/which_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(pt_dir, fid)


100%|██████████| 50001/50001 [00:27<00:00, 1803.17it/s]


samplings/GMDiT/1.4/3/Dual-Solver/50000/which_19/which_0 8.492323868160895


In [4]:
pt_dirs = [ 
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_0/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_1/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_2/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_3/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_4/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_5/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_6/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_7/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_8/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_9/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_10/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_11/which_0',
            # 'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_12/which_0',
            # 'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_13/which_0',
            # 'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_14/which_0',
            # 'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_15/which_0',
            # 'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_16/which_0',
            # 'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_17/which_0',
            # 'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_18/which_0',
            'samplings/GMDiT/1.4/9/Dual-Solver/50000/which_19/which_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(pt_dir, fid)


100%|██████████| 50001/50001 [00:32<00:00, 1552.63it/s]


samplings/GMDiT/1.4/9/Dual-Solver/50000/which_19/which_0 3.8674655726375136


In [32]:
pt_dirs = [ #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_0/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_1/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_2/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_3/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_4/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_5/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_6/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_7/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_8/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_9/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_10/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_11/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_12/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_13/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_14/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_15/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_16/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_17/which_0',
            #'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_18/which_0',
            'samplings/GMDiT/1.4/9/Dual-Solver/10000/which_19/which_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    try:    
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)
    except:
        continue


  0%|          | 0/10001 [00:00<?, ?it/s]

100%|██████████| 10001/10001 [00:04<00:00, 2126.31it/s]


samplings/GMDiT/1.4/9/Dual-Solver/10000/which_19/which_0 6.223178333390877


In [13]:
pt_dirs = [ 'samplings/GMDiT/1.4/9/Dual-Solver/50000/traj_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    try:    
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], num=10000)
        print(pt_dir, fid)
    except:
        continue


  0%|          | 0/10000 [00:00<?, ?it/s]

100%|██████████| 10000/10000 [00:03<00:00, 2551.25it/s]


samplings/GMDiT/1.4/9/Dual-Solver/50000/traj_0 5.292212326967217


In [3]:
pt_dirs = [ #'samplings/GMDiT/1.4/5/Dual-Solver/50000/3/dual_0',
            #'samplings/GMDiT/1.4/7/Dual-Solver/50000/3/dual_0',
            'samplings/GMDiT/1.4/9/Dual-Solver/50000/3/dual_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    try:    
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)
    except:
        continue


 16%|█▌        | 8005/50001 [00:03<00:18, 2298.98it/s]


In [9]:
pt_dirs = [ #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_0/which_0',
            #samplings/GMDiT/1.4/3/Dual-Solver/10000/which_1/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_2/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_3/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_4/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_5/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_6/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_7/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_8/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_9/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_10/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_11/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_12/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_13/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_14/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_15/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_16/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_17/which_0',
            #'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_18/which_0',
            'samplings/GMDiT/1.4/3/Dual-Solver/10000/which_19/which_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    try:    
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)
    except:
        continue


100%|██████████| 10001/10001 [00:04<00:00, 2108.74it/s]


samplings/GMDiT/1.4/3/Dual-Solver/10000/which_19/which_0 11.204905034935507


In [22]:
pt_dirs = [ #'samplings/GMDiT/1.4/3/Dual-Solver/50000/traj_0',
            # 'samplings/GMDiT/1.4/5/Dual-Solver/50000/traj_0',
            # 'samplings/GMDiT/1.4/7/Dual-Solver/50000/traj_0',
            'samplings/GMDiT/1.4/8/Dual-Solver/50000/traj_0',
            # 'samplings/GMDiT/1.4/9/Dual-Solver/50000/traj_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    try:    
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)
    except:
        continue


100%|██████████| 50001/50001 [00:27<00:00, 1807.16it/s]


samplings/GMDiT/1.4/8/Dual-Solver/50000/traj_0 2.4365456228673565


In [8]:
pt_dirs = [#'samplings/GMDiT/1.4/9/DS-Solver_Flow/50000/ds_0/',
           'samplings/GMDiT/1.4/7/DS-Solver_Flow/50000/ds_0/',
            #'samplings/GMDiT/1.4/5/DS-Solver_Flow/50000/ds_0/',
            #'samplings/GMDiT/1.4/3/DS-Solver_Flow/50000/ds_0/',
            ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], num=46001)
    print(config.solver, config.NFE, config.CFG, fid)


  0%|          | 0/46001 [00:00<?, ?it/s]

100%|██████████| 46001/46001 [00:24<00:00, 1851.42it/s]


DS-Solver_Flow 7 1.4 2.7547168487761837


In [6]:
pt_dirs = [ 'samplings/GMDiT/1.4/9/Dual-Solver/50000/1/random_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    try:    
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)
    except:
        continue


100%|██████████| 50001/50001 [00:27<00:00, 1830.87it/s]


samplings/GMDiT/1.4/9/Dual-Solver/50000/1/random_0 5.951159587559857


In [5]:
pt_dirs = [ 'samplings/GMDiT/1.4/9/Dual-Solver/50000/1/dual_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    try:    
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], num=21558)
        print(pt_dir, fid)
    except:
        continue


100%|██████████| 21558/21558 [00:10<00:00, 2082.65it/s]


samplings/GMDiT/1.4/9/Dual-Solver/50000/1/dual_0 4.195276753483029


In [24]:
pt_dirs = [ 'samplings/GMDiT/1.4/3/Dual-Solver_Quad/50000/dual_0',
            'samplings/GMDiT/1.4/5/Dual-Solver_Quad/50000/dual_0',
            'samplings/GMDiT/1.4/7/Dual-Solver_Quad/50000/dual_0',
            'samplings/GMDiT/1.4/9/Dual-Solver_Quad/50000/dual_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    try:    
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)
    except:
        continue


  1%|          | 348/50001 [00:00<00:37, 1330.76it/s]

100%|██████████| 50001/50001 [00:38<00:00, 1286.17it/s]


samplings/GMDiT/1.4/3/Dual-Solver_Quad/50000/dual_0 9.382812545179263


100%|██████████| 50001/50001 [00:25<00:00, 1942.56it/s]


samplings/GMDiT/1.4/5/Dual-Solver_Quad/50000/dual_0 4.006655867242102


100%|██████████| 50001/50001 [00:25<00:00, 1934.64it/s]


samplings/GMDiT/1.4/7/Dual-Solver_Quad/50000/dual_0 3.436872483350271


100%|██████████| 50001/50001 [00:26<00:00, 1905.26it/s]


samplings/GMDiT/1.4/9/Dual-Solver_Quad/50000/dual_0 3.0420566967887908


In [25]:
pt_dirs = [ 'samplings/GMDiT/1.4/3/Dual-Solver_Legendre/50000/dual_0',
            'samplings/GMDiT/1.4/5/Dual-Solver_Legendre/50000/dual_0',
            'samplings/GMDiT/1.4/7/Dual-Solver_Legendre/50000/dual_0',
            'samplings/GMDiT/1.4/9/Dual-Solver_Legendre/50000/dual_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    try:    
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)
    except:
        continue


100%|██████████| 50001/50001 [00:33<00:00, 1475.67it/s]


samplings/GMDiT/1.4/3/Dual-Solver_Legendre/50000/dual_0 8.271479226313545


100%|██████████| 50001/50001 [00:37<00:00, 1340.14it/s]


samplings/GMDiT/1.4/5/Dual-Solver_Legendre/50000/dual_0 3.70181789281429


100%|██████████| 50001/50001 [00:23<00:00, 2148.90it/s]


samplings/GMDiT/1.4/7/Dual-Solver_Legendre/50000/dual_0 3.970312571989666


100%|██████████| 50001/50001 [00:23<00:00, 2131.16it/s]


samplings/GMDiT/1.4/9/Dual-Solver_Legendre/50000/dual_0 3.660210747282349


In [26]:
pt_dirs = [ 'samplings/GMDiT/1.4/3/Dual-Solver/50000/3/dual_0/',
            'samplings/GMDiT/1.4/5/Dual-Solver/50000/3/dual_0/',
            'samplings/GMDiT/1.4/7/Dual-Solver/50000/3/dual_0/',
            'samplings/GMDiT/1.4/9/Dual-Solver/50000/3/dual_0/',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    try:    
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)
    except:
        continue


100%|██████████| 50001/50001 [00:22<00:00, 2225.24it/s]


samplings/GMDiT/1.4/3/Dual-Solver/50000/3/dual_0/ 7.589750084109767


100%|██████████| 50001/50001 [00:24<00:00, 2030.94it/s]


samplings/GMDiT/1.4/5/Dual-Solver/50000/3/dual_0/ 3.598641944252563


100%|██████████| 50001/50001 [00:23<00:00, 2122.93it/s]


samplings/GMDiT/1.4/7/Dual-Solver/50000/3/dual_0/ 3.289776186818699


100%|██████████| 50001/50001 [00:16<00:00, 2997.58it/s]


samplings/GMDiT/1.4/9/Dual-Solver/50000/3/dual_0/ 3.249724066144836


In [66]:
pt_dirs = ['samplings/GMDiT/1.4/9/Dual-Solver/50000/3/dual_0',
           'samplings/GMDiT/1.4/9/Dual-Solver/50000/5/dual_0',
           'samplings/GMDiT/1.4/9/Dual-Solver/50000/7/dual_0',
           'samplings/GMDiT/1.4/9/Dual-Solver/50000/9/dual_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    try:    
        data = torch.load('/dataset/dit/stats.pt')
        fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
        print(pt_dir, fid)
    except:
        continue


  0%|          | 0/18481 [00:00<?, ?it/s]

100%|██████████| 18481/18481 [00:10<00:00, 1736.64it/s]


samplings/GMDiT/1.4/9/Dual-Solver/50000/3/dual_0 4.286049027690694


In [4]:
pt_dirs = ['samplings/GMDiT/1.4/3/BNS-Solver/50000/bns_0/',
            'samplings/GMDiT/1.4/5/BNS-Solver/50000/bns_0/',
            'samplings/GMDiT/1.4/7/BNS-Solver/50000/bns_0/',
            'samplings/GMDiT/1.4/9/BNS-Solver/50000/bns_0/',
            ]
for pt_dir in pt_dirs:
    #config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print(pt_dir, fid)


100%|██████████| 50001/50001 [00:34<00:00, 1438.92it/s]


samplings/GMDiT/1.4/3/BNS-Solver/50000/bns_0/ 57.885618505615355


 61%|██████▏   | 30741/50001 [00:16<00:10, 1847.70it/s]


KeyboardInterrupt: 

In [16]:
pt_dirs = [#'samplings/GMDiT/1.4/9/DS-Solver_Flow/50000/ds_0/',
           'samplings/GMDiT/1.4/7/DS-Solver_Flow/50000/ds_0/',
            #'samplings/GMDiT/1.4/5/DS-Solver_Flow/50000/ds_0/',
            #'samplings/GMDiT/1.4/3/DS-Solver_Flow/50000/ds_0/',
            ]
for pt_dir in pt_dirs:
    config = load_config(pt_dir)
    data = torch.load('/dataset/dit/stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], num=39821)
    print(config.solver, config.NFE, config.CFG, fid)


100%|██████████| 39821/39821 [00:55<00:00, 715.23it/s]


DS-Solver_Flow 7 1.4 2.871137630044302
